# 04 — Model training

**This notebook reads saved results.** Training itself is `ml_pipeline/train.py` so we do not fit models twice by accident.

## What did we train?

One-hour-ahead Spanish demand (`load_mw` at hour t).

## Split (no shuffling)

| Split | Period | Why |
| --- | --- | --- |
| Train | through 2018 | Fit models |
| Validation | 2019 | Choose the winner |
| Test | 2020 | Report performance, including COVID demand shift |

2020 was **not** used to pick the model. If test MAE is worse than validation MAE, that is expected, not a bug we should “fix” by training on 2020.

In [1]:
from pathlib import Path
import json
import pandas as pd

def project_root() -> Path:
    here = Path.cwd()
    for candidate in (here, here.parent):
        if (candidate / "models" / "model_metadata.json").exists():
            return candidate
    raise FileNotFoundError("Run: python ml_pipeline/train.py")

ROOT = project_root()
metadata = json.loads((ROOT / "models" / "model_metadata.json").read_text(encoding="utf-8"))
print("Selected:", metadata["selected_model"])
print("Rule:", metadata["selection_rule"])
print(metadata["split"])

Selected: hist_gradient_boosting
Rule: lowest validation MAE; test set was not used to choose
{'train': 'utc < 2019-01-01', 'valid': '2019-01-01 <= utc < 2020-01-01', 'test': 'utc >= 2020-01-01 (COVID demand shift)', 'shuffled': False, 'n_train': 34679, 'n_valid': 8760, 'n_test': 6575}


## Models compared

1. **Persistence** — predict `lag_1` (last hour). Strong baseline from EDA.
2. **Moving average 24** — predict `rolling_mean_24`. Weaker baseline.
3. **Linear regression** — scaled features, still a linear combination.
4. **Random forest** — nonlinear, sklearn.
5. **Histogram gradient boosting** — sklearn boosting. XGBoost was not added because this already covers gradient boosting without another library.

Winner = **lowest validation MAE**.

In [2]:
rows = []
for name, scores in metadata["metrics"].items():
    rows.append(
        {
            "model": name,
            "valid_MAE_MW": scores["valid"]["mae"],
            "valid_RMSE_MW": scores["valid"]["rmse"],
            "valid_MAPE_pct": scores["valid"]["mape"],
            "test_MAE_MW": scores["test"]["mae"],
            "test_RMSE_MW": scores["test"]["rmse"],
            "test_MAPE_pct": scores["test"]["mape"],
        }
    )
table = pd.DataFrame(rows).sort_values("valid_MAE_MW")
table

,model,valid_MAE_MW,valid_RMSE_MW,valid_MAPE_pct,test_MAE_MW,test_RMSE_MW,test_MAPE_pct
4,hist_gradient_boosting,227.02,313.49,0.801,311.33,443.58,1.237
3,random_forest,273.48,407.84,0.957,398.84,583.19,1.555
2,linear_regression,765.46,1021.77,2.715,733.92,968.65,2.747
0,persistence,1030.20,1350.88,3.677,1006.85,1275.53,3.796
1,moving_average_24,3254.23,3841.73,11.911,3019.60,3606.56,11.600


These numbers come from `models/model_metadata.json`, which was written by the training script. They are not placeholders.

Histogram gradient boosting won on 2019 validation. Test MAE is higher because 2020 demand dropped (COVID). Persistence is already ~1,030 MW MAE — any fancier model must beat that, and the trees did.